In [5]:
base_path = '/kaggle/input/datasets/almutalem/rehab24-6/REHAB24-6'

In [6]:
!ls -l {base_path}

total 68
drwxr-xr-x 8 nobody nogroup     0 Sep  4 09:33 2d_joints
drwxr-xr-x 8 nobody nogroup     0 Sep  4 09:33 2d_markers
drwxr-xr-x 8 nobody nogroup     0 Sep  4 09:34 3d_joints
drwxr-xr-x 8 nobody nogroup     0 Sep  4 09:34 3d_markers
-rw-r--r-- 1 nobody nogroup   395 Sep  4 09:34 joints_names.txt
-rw-r--r-- 1 nobody nogroup   581 Sep  4 09:34 marker_names.txt
-rw-r--r-- 1 nobody nogroup 53290 Sep  4 09:34 Segmentation.csv
-rw-r--r-- 1 nobody nogroup  1729 Sep  4 09:34 Segmentation.txt


In [7]:
with open(f'{base_path}/joints_names.txt', 'r') as file:
    print(file.read())

 0: Hips
 1: Spine
 2: Spine1
 3: Neck
 4: Head
 5: Head_end
 6: LeftShoulder
 7: LeftArm
 8: LeftForeArm
 9: LeftHand
10: LeftHand_end
11: RightShoulder
12: RightArm
13: RightForeArm
14: RightHand
15: RightHand_end
16: LeftUpLeg
17: LeftLeg
18: LeftFoot
19: LeftToeBase
20: LeftToeBase_end
21: RightUpLeg
22: RightLeg
23: RightFoot
24: RightToeBase
25: RightToeBase_end


In [8]:
with open(f'{base_path}/marker_names.txt', 'r') as file:
    print(file.read())

 0: BackLeft
 1: BackRight
 2: BackTop
 3: Chest
 4: HeadFront
 5: HeadSide
 6: HeadTop
 7: LAnkleOut
 8: LElbowOut
 9: LHandOut
10: LHeel
11: LKneeOut
12: LShin
13: LShoulderBack
14: LShoulderTop
15: LThigh
16: LToeIn
17: LToeOut
18: LToeTip
19: LUArmHigh
20: LWristIn
21: LWristOut
22: RAnkleOut
23: RElbowOut
24: RHandOut
25: RHeel
26: RKneeOut
27: RShin
28: RShoulderBack
29: RShoulderTop
30: RThigh
31: RToeIn
32: RToeOut
33: RToeTip
34: RUArmHigh
35: RWristIn
36: RWristOut
37: WaistLBack
38: WaistLFront
39: WaistRBack
40: WaistRFront


In [9]:
with open(f'{base_path}/Segmentation.txt', 'r') as file:
    print(file.read())

Semantics of csv file columns and values
video_id -- id of the video file
repetition_number -- number of repetition within given video
exercise_id -- id of the exercise; description of exercises is provided on the main dataset page on Zenodo
person_id -- id of exercising person
first_frame -- number of the first frame of the repetition
last_frame -- number of the last frame of the repetition
cam17_orientation -- orientation of exercising person towards camera17, possible values are 'front', 'half-profile' and 'profile'; as camera18 was placed orthogonally to camera17, this also implies orientation of the person towards camera18
  cam17 orientation -> cam18 orientation
  front -> side
  half-profile -> half-profile
  side -> front
mocap_erroneous -- indication whether there is an error in mocap data (some markers not correctly detected)
exercise_subtype -- for some exercises, we distinguish between right- and left-sided execution of the exercise
lights_on -- indicates whether lights wer

- Ex1 = Arm abduction: sideway raising of the straightened right arm;
- Ex2 = Arm VW: fluent transition of arms between V (arms straight up) and W (elbows down, hands up) shape;
- Ex3 = Push-ups: push-ups with hands on a table;
- Ex4 = Leg abduction: sideway raising of the straightened leg;
- Ex5 = Leg lunge: pushing a knee of the back leg down while keeping a right angle on the front knee;
- Ex6 = Squats.

In [10]:
import pandas as pd

In [11]:
segment = pd.read_csv(f'{base_path}/Segmentation.csv', sep=';')

In [12]:
segment.head()

,video_id,repetition_number,exercise_id,person_id,first_frame,last_frame,cam17_orientation,mocap_erroneous,exercise_subtype,lights_on,extra_person_in_cam17,extra_person_in_cam18,correctness
0,PM_000,1,1,1,180,377,front,0,right arm,0,3,0,1
1,PM_000,2,1,1,378,620,front,0,right arm,0,3,0,1
2,PM_000,3,1,1,621,865,front,0,right arm,0,3,0,1
3,PM_000,4,1,1,866,1085,front,0,right arm,0,3,3,1
4,PM_000,5,1,1,1086,1265,front,0,right arm,0,3,3,1


### The dataset contains the recording of the exercise in two different pre-extracted formats:
- **2D**: This contains the recording of the person in 2D format contains the recording in two different orthogonal plane (`C17` and `C18`).
- **3D**: This contains teh recording of the person in 3D plane. 

In [17]:
import numpy as np

data = np.load(f'{base_path}/2d_joints/Ex1/PM_000-c17-120fps.npy')
data.shape

(5215, 26, 2)

In [18]:
data = np.load(f'{base_path}/2d_joints/Ex1/PM_000-c17-30fps.npy')
data.shape

(1304, 26, 2)

- For each exercise multiple person has performed it and in two differnt fps is available (`120fps` & `30fps`).
- The each **2D** data has two orthogonal view (`C17` and `C18`).
- Each recording contains `5215 for 120fps` and `1304 for 30fps`.
- For both `2D` and `3D` data there exist two different extracted data `joints` and `markers`.
- **Markers**: These are the actual points that are being tracked during the recording.
- **Joints**: These are extracted points based on `markers` that shows the human joints.

Going to continue with the `2D 30fps` data. As the consumer hardware will be able to only extract the 2D frame properly and can suffer with the proper depth and space position, and the reason for opting for the 30fps is also due to the limited consumer hardware capability. Hence to make the model train in the similar type of data we are opting for the above choice instead of going with 3D or the 120fps options.